## ENEM MG 2021–2024 — Consolidação longitudinal e agregações

Objetivo: consolidar os recortes de Minas Gerais entre 2021 e 2024, harmonizar variáveis de participantes e resultados, gerar tabelas agregadas para análise exploratória e dashboard, e construir bases integradas em nível agregado para comparações entre perfil socioeconômico e desempenho.

Entradas:

* DADOS_TRAT_MG_2021
* DADOS_TRAT_MG_2022
* DADOS_TRAT_MG_2023
* DADOS_TRAT_MG_2024
* RESULTADOS_TRAT_MG_2024

Saídas esperadas:

* base consolidada MG 2021–2024 em nível candidato;
* base consolidada de resultados;
* base agregada socioeconômica;
* base agregada da base de resultados;
* base merged agregada;
* amostra estratificada de resultados;
* base numérica consolidada para modelagem;
* base merged numérica.
  
Observação metodológica: nos anos de 2021 a 2023, as informações socioeconômicas e de desempenho estão presentes na mesma base tratada. Em 2024, participantes e resultados foram divulgados separadamente, o que impede associação individual. Assim, a integração entre perfil socioeconômico e desempenho em 2024 é feita apenas em nível agregado.

Observação (GitHub): esta etapa trabalha apenas sobre arquivos Parquet previamente tratados e harmonizados nas etapas anteriores.

### 1) Ambiente e imports

In [1]:
import sys
from pathlib import Path

# Permite importar o pacote `src/` a partir do diretório do projeto.
ROOT_PATH = Path().resolve().parents[1]  # notebooks/00_preprocessamento -> projeto
if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

import re
import numpy as np
import pandas as pd


from src.config import (
    DADOS_TRAT_MG_2021,
    DADOS_TRAT_MG_2022,
    DADOS_TRAT_MG_2023,
    DADOS_TRAT_MG_2024,
    RESULTADOS_TRAT_MG_2024,
    DADOS_MG_21_23_NUM,
    DADOS_MG_21_23,
    DADOS_AGG_MG_21_23, 
    MERGED_MG,
    MERGED_MG_NUM,
    DADOS_AGG_MG,
    RESULTADOS_AGG_MG,
    DADOS_UNI_MG,
    RESULTADOS_UNI_MG,
    AMOSTRAG_RESULTADOS_MG,
)
    

from src.preprocessamento.agregacoes import  (
    agregar_perfil_socioeconomico,
    agrupar_notas, 
    amostrar_por_percentil_original, 
    categoria_ordenada_para_numero,
    numero_para_categoria,
)

from src.preprocessamento.categorias import (
    ORDEM_OCUPACAO, 
    ORDEM_PAIS_ESCOLARIDADE, 
    MAP_ESCOLARIDADE_REV,
    MAP_OCUP_REV,
)




pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")


### 2) Leitura e consolidação anual

Nesta etapa são carregados os recortes anuais de Minas Gerais já tratados. Em seguida, é criada a variável ano e os arquivos são concatenados para formar uma base longitudinal única, utilizada nas análises exploratórias, nas agregações e na preparação das bases para modelagem.

In [2]:
df_mg_21  = pd.read_parquet(DADOS_TRAT_MG_2021)
df_mg_22  = pd.read_parquet(DADOS_TRAT_MG_2022)
df_mg_23  = pd.read_parquet(DADOS_TRAT_MG_2023)
df_mg_24  = pd.read_parquet(DADOS_TRAT_MG_2024)
df_mg_r_24  = pd.read_parquet(RESULTADOS_TRAT_MG_2024)


In [3]:
df_mg_21['ano'] = '2021'
df_mg_22['ano'] = '2022'
df_mg_23['ano'] = '2023'
df_mg_24['ano'] = '2024'
df_mg_r_24['ano'] = '2024'

df_mg_21_23 = pd.concat([df_mg_21, df_mg_22, df_mg_23], ignore_index=True)

for base in [df_mg_21_23, df_mg_24, df_mg_r_24]:
    base["ano"] = base["ano"].astype(str)
    base.drop(columns=["municipio"], axis=1, inplace=True, errors="ignore")

In [4]:
df_mg_21_23.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media,regiao,ano
0,20-25,feminino,solteiro,branca,não informada,MG,NaN,574.60,472.60,NaN,espanhol,760.00,superior,superior,básico,básico,3,1 a 3,0,1,1,1,1,1,0.18,2.00,602.40,Metrop. de Belo Horizonte,2021
1,20-25,feminino,solteiro,negra,pública,MG,487.40,476.50,450.70,493.40,inglês,520.00,até fund,até fund,básico,básico,2,1 a 3,0,1,0,1,1,0,0.15,2.00,485.60,Metrop. de Belo Horizonte,2021
2,20-25,masculino,solteiro,negra,não informada,MG,582.30,664.30,576.00,528.70,inglês,580.00,até fund,médio,básico,básico,3,1 a 3,5,1,1,3,3,2,0.32,2.00,586.26,Metrop. de Belo Horizonte,2021


### 3) Harmonização entre base de dados e base de resultados

Como os anos de 2021 a 2023 contêm informações socioeconômicas e de desempenho na mesma base, enquanto 2024 traz participantes e resultados em arquivos separados, foi necessário harmonizar a estrutura dos dados em dois blocos:

* base de resultados, contendo notas e métricas de presença;
* base de dados, contendo características socioeconômicas e demográficas.

Essa separação permite gerar agregações compatíveis entre os anos e construir posteriormente bases integradas em nível agregado.

In [5]:
colunas_notas = [col for col in df_mg_21_23.columns if col.startswith("nota")]
colunas_notas.append("lingua")

colunas_resultados = [
    "ano", "escola", "uf", "regiao",
    "nota_cn", "nota_ch", "nota_lc", "nota_mt",
    "lingua", "nota_redacao", "nota_media"
]

df_result_21_23 = df_mg_21_23[colunas_resultados].copy()

df_resultados = pd.concat([df_result_21_23, df_mg_r_24], ignore_index=True)
df_resultados = df_resultados.drop(columns=["municipio"], errors="ignore")

df_dados_21_23 = df_mg_21_23.drop(columns=colunas_notas, errors="ignore").copy()
df_dados = pd.concat([df_dados_21_23, df_mg_24], ignore_index=True)

### 4) Agregação da base de resultados


A base de resultados é agregada para sintetizar o desempenho médio dos grupos e os indicadores de comparecimento às provas. Essa agregação permite comparar regiões e tipos de escola sem depender de vínculo individual entre participantes e resultados.

São calculados:

* médias de notas por área e da nota_media;
* maiores notas observadas por área;
* desvio-padrão da nota_media;
* quantidade de inscritos;
* faltosos e taxas de presença por dia de prova.

In [6]:
df_mg_r_24_agg = agrupar_notas(df_mg_r_24, incluir_regiao=True)
df_resultados_agg = agrupar_notas(df_resultados, incluir_regiao=True)

df_resultados_agg.head(3)


,ano,uf,escola,regiao,nota_media,nota_cn,nota_cn_max,nota_ch,nota_ch_max,nota_lc,nota_lc_max,nota_mt,nota_mt_max,nota_redacao,nota_redacao_max,desvio_padrao,participantes,inscritos,faltosos_dia1,faltosos_dia2,presentes_dia1,presentes_dia2,taxa_presenca_dia1,taxa_presenca_dia2
0,2021,MG,pública,Campo das Vertentes,545.22,497.58,789.30,528.42,778.00,511.51,736.80,555.12,927.10,637.88,960.00,87.46,1880,1880,387,457,1493,1423,0.79,0.76
1,2021,MG,pública,Centro de Minas,531.21,486.37,720.30,511.53,814.60,499.66,708.10,535.98,818.20,628.28,980.00,90.16,1712,1712,362,419,1350,1293,0.79,0.76
2,2021,MG,pública,Metrop. de Belo Horizonte,537.23,492.58,848.70,526.60,832.50,511.28,746.70,544.64,948.80,617.68,980.00,85.78,24520,24520,5647,6591,18873,17929,0.77,0.73


### 5) Preparação da versão numérica para modelagem

Para a etapa de modelagem, também foi criada uma versão numérica da base consolidada. Nessa versão, variáveis ordinais relacionadas ao capital familiar — como escolaridade e ocupação dos pais — são convertidas em códigos numéricos, preservando a ordem substantiva definida no pré-processamento.

Essa conversão é útil para:

* correlação;
* análises de sensibilidade;
*construção de bases merged para modelagem;
* consolidação final do dataset voltado a ML.

### 5.1) Confirmação da ordenação categórica

In [7]:
print(df_dados['ocup_pai'].unique())
print(df_dados['escolaridade_pai'].unique())

['básico', 'rural', 'desconhecida', 'manual/tec', 'médio/tec', 'superior']
Categories (6, object): ['desconhecida' < 'rural' < 'básico' < 'manual/tec' < 'médio/tec' < 'superior']
['superior', 'até fund', 'desconhecida', 'médio', 'não estudou', 'pós-grad']
Categories (6, object): ['desconhecida' < 'não estudou' < 'até fund' < 'médio' < 'superior' < 'pós-grad']


### 5.2) Conversão para códigos numéricos

Transformação da variável escola para análise de correlação

A variável escola é originalmente categórica nominal, representando o tipo de instituição de ensino médio frequentada pelos participantes:

* pública
* privada
* não informada

Para viabilizar sua inclusão em análises de correlação, foi adotada uma codificação ordinal controlada, baseada na relação empírica observada entre tipo de escola e desempenho médio.

Observou-se que:

* candidatos de escolas públicas apresentam, em média, menores notas;
* candidatos que não informaram o tipo de escola apresentam desempenho intermediário;
* candidatos de escolas privadas apresentam, em média, maiores notas.

Com base nesse padrão, definiu-se a seguinte ordenação:

* pública → 0
* não informada → 1
* privada → 2

Essa transformação não implica que a variável seja quantitativa, mas permite sua utilização como uma aproximação ordinal em análises exploratórias, especialmente em matrizes de correlação.

Interpretação

A codificação deve ser interpretada como um gradiente aproximado de condições educacionais associadas ao tipo de escola, e não como uma medida contínua.

Limitação metodológica

Essa abordagem pressupõe uma relação monotônica entre as categorias, o que pode não capturar completamente a heterogeneidade interna de cada grupo.

Por esse motivo, na etapa de modelagem, a variável escola é tratada como categórica, sendo transformada por meio de one-hot encoding, evitando a imposição de uma estrutura ordinal artificial no modelo.

In [8]:
df_dados_num = df_dados.copy()
df_mg_21_23_num = df_mg_21_23.copy()
dfs_num=[df_dados_num, df_mg_21_23_num]

for col in ["ocup_pai", "ocup_mae", "escolaridade_pai", "escolaridade_mae"]:
    df_dados_num[col] = categoria_ordenada_para_numero(df_dados_num[col])
    df_mg_21_23_num[col] = categoria_ordenada_para_numero(df_mg_21_23_num[col])

for base in dfs_num:
    base ['escola_num']=categoria_ordenada_para_numero(base['escola'])
    base['escolaridade_pais_media'] = (base['escolaridade_pai'] + base['escolaridade_mae']) / 2
    base['ocup_pais_media'] = (base['ocup_mae'] + base['ocup_pai']) / 2

df_mg_21_23_num.head(3)

,faixa_etaria,sexo,estado_civil,cor_raca,escola,uf,nota_cn,nota_ch,nota_lc,nota_mt,lingua,nota_redacao,escolaridade_pai,escolaridade_mae,ocup_pai,ocup_mae,n_pessoas_resd,sal_min,emp_domst,gelad,lv_rp,tv,cel,comptdr,indice_consumo,renda_media,nota_media,regiao,ano,escola_num,escolaridade_pais_media,ocup_pais_media
0,20-25,feminino,solteiro,branca,não informada,MG,NaN,574.60,472.60,NaN,espanhol,760.00,4,4,2,2,3,1 a 3,0,1,1,1,1,1,0.18,2.00,602.40,Metrop. de Belo Horizonte,2021,1,4.00,2.00
1,20-25,feminino,solteiro,negra,pública,MG,487.40,476.50,450.70,493.40,inglês,520.00,2,2,2,2,2,1 a 3,0,1,0,1,1,0,0.15,2.00,485.60,Metrop. de Belo Horizonte,2021,0,2.00,2.00
2,20-25,masculino,solteiro,negra,não informada,MG,582.30,664.30,576.00,528.70,inglês,580.00,2,3,2,2,3,1 a 3,5,1,1,3,3,2,0.32,2.00,586.26,Metrop. de Belo Horizonte,2021,1,2.50,2.00


### 6) Amostra estratificada de resultados

Além da base completa agregada, foi criada uma amostra estratificada da base de resultados, distribuída ao longo dos percentis de nota_media. Essa amostra reduz o volume da base preservando a heterogeneidade do desempenho e pode ser útil em visualizações, testes e análises complementares.

In [9]:
df_amostra_resultados = amostrar_por_percentil_original(
    df_resultados,
    q=20,
    n_por_percentil=100
)
print(f"Amostra com distribuição original: {len(df_amostra_resultados)} linhas")

Amostra com distribuição original: 246365 linhas


### 7) Agregação da base socioeconômica

A base socioeconômica é agregada por ano, região e características demográficas e familiares. O objetivo é construir perfis médios de grupos populacionais comparáveis ao longo do tempo, preservando tanto o tamanho dos grupos quanto indicadores de infraestrutura, renda e composição domiciliar.

In [10]:
cols_agg = [
    "ano",
    "uf",
    "regiao",
    "escola",
    "sexo",
    "cor_raca",
    "faixa_etaria",
    "sal_min",
    "escolaridade_pai",
    "escolaridade_mae",
    "ocup_pai",
    "ocup_mae",
]

df_dados_agg = agregar_perfil_socioeconomico(df_dados, cols_agg)
df_dados_agg.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 407079 entries, 0 to 407078
Data columns (total 18 columns):
 #   Column            Non-Null Count   Dtype   
---  ------            --------------   -----   
 0   ano               407079 non-null  object  
 1   uf                407079 non-null  category
 2   regiao            407079 non-null  category
 3   escola            407079 non-null  category
 4   sexo              407079 non-null  category
 5   cor_raca          407079 non-null  category
 6   faixa_etaria      407079 non-null  category
 7   sal_min           407079 non-null  category
 8   escolaridade_pai  407079 non-null  category
 9   escolaridade_mae  407079 non-null  category
 10  ocup_pai          407079 non-null  category
 11  ocup_mae          407079 non-null  category
 12  participantes     407079 non-null  int64   
 13  cel               407079 non-null  Float64 
 14  comptdr           407079 non-null  Float64 
 15  n_pessoas_resd    407079 non-null  Float64 
 16  re

### 8) Base merged agregada
Para analisar conjuntamente indicadores socioeconômicos e desempenho educacional, foi construída uma base integrada em nível agregado. Como a associação individual não é possível em 2024, a integração é realizada por chaves comuns (ano, uf, regiao, escola), unindo a agregação da base de dados à agregação da base de resultados.

In [11]:
cols_merged = ["ano", "uf", "regiao", "escola"]

df_dados_merged = agregar_perfil_socioeconomico(df_dados_num, cols_merged, incluir_escola_num_pais_media=True, )

df_merged = pd.merge(
    df_resultados_agg,
    df_dados_merged,
    on=["ano", "uf", "regiao", "escola"],
    how="outer",
    suffixes=("_notas", "_renda"),
)


df_merged = df_merged.drop(columns=["participantes_notas"], errors="ignore")
df_merged= df_merged.rename(columns={"participantes_renda": "participantes"})
df_merged.head(3)

,ano,uf,escola,regiao,nota_media,nota_cn,nota_cn_max,nota_ch,nota_ch_max,nota_lc,nota_lc_max,nota_mt,nota_mt_max,nota_redacao,nota_redacao_max,desvio_padrao,inscritos,faltosos_dia1,faltosos_dia2,presentes_dia1,presentes_dia2,taxa_presenca_dia1,taxa_presenca_dia2,participantes,cel,comptdr,n_pessoas_resd,renda_media,indice_consumo,escola_num,escolaridade_pais_media,ocup_pais_media
0,2021,MG,pública,Campo das Vertentes,545.22,497.58,789.30,528.42,778.00,511.51,736.80,555.12,927.10,637.88,960.00,87.46,1880,387,457,1493,1423,0.79,0.76,1880,2.87,0.92,3.83,2.90,0.29,0.00,2.83,2.53
1,2021,MG,não informada,Campo das Vertentes,556.78,512.12,765.00,545.33,846.90,521.27,720.00,567.77,948.80,644.63,980.00,89.93,5821,1910,2143,3911,3678,0.67,0.63,5821,2.65,1.00,3.52,3.01,0.28,1.00,2.72,2.51
2,2021,MG,privada,Campo das Vertentes,632.58,563.46,733.70,590.61,761.80,566.96,731.70,651.49,900.70,792.34,980.00,79.75,266,5,9,261,257,0.98,0.97,266,3.20,1.87,3.70,6.51,0.41,2.00,3.82,3.69


### 9) Base histórica agregada 2021–2023

Além da consolidação longitudinal completa, foi gerada uma agregação específica para o período 2021–2023, em que informações socioeconômicas e notas estavam presentes no mesmo arquivo-base. Isso permite comparações históricas diretas sem necessidade de integração entre arquivos distintos.

In [12]:
cols_agg=['ano','uf', 'regiao', 'escola', 'sal_min','cor_raca', 'faixa_etaria','escolaridade_pai', 'escolaridade_mae', 'ocup_pai',
       'ocup_mae']

for col in cols_agg:
    if df_mg_21_23_num[col].dtype.name == 'category':
        df_mg_21_23_num[col] = df_mg_21_23_num[col].cat.remove_unused_categories()

df_agg_21_23 = df_mg_21_23_num.groupby(cols_agg, observed=True, dropna=False).agg(
    participantes=("regiao", "count"),
    cel=("cel", lambda x: round(x.mean())),
    comptdr=("comptdr", lambda x: round(x.mean())),
    n_pessoas_resd=("n_pessoas_resd", lambda x: round(x.mean())),
    renda_media=("renda_media","mean"),
    indice_consumo=("indice_consumo", "mean"),
    escolaridade_pais_media= ("escolaridade_pais_media", "mean"),
    ocup_pais_media= ("ocup_pais_media", "mean"),
    nota_cn=("nota_cn", "mean"),
    nota_ch=("nota_ch", "mean"),
    nota_lc=("nota_lc", "mean"),
    nota_mt=("nota_mt", "mean"),
    nota_redacao=("nota_redacao", "mean"),
    nota_media=("nota_media", "mean"),
).round(2)

df_agg_21_23 = df_agg_21_23.reset_index()

# Filtrar apenas grupos com participantes
df_agg_21_23 = df_agg_21_23[df_agg_21_23['participantes'] > 0]

df_agg_21_23.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231748 entries, 0 to 231747
Data columns (total 25 columns):
 #   Column                   Non-Null Count   Dtype   
---  ------                   --------------   -----   
 0   ano                      231748 non-null  object  
 1   uf                       231748 non-null  category
 2   regiao                   231748 non-null  category
 3   escola                   231748 non-null  category
 4   sal_min                  231748 non-null  category
 5   cor_raca                 231748 non-null  category
 6   faixa_etaria             231748 non-null  category
 7   escolaridade_pai         231748 non-null  Int16   
 8   escolaridade_mae         231748 non-null  Int16   
 9   ocup_pai                 231748 non-null  Int16   
 10  ocup_mae                 231748 non-null  Int16   
 11  participantes            231748 non-null  int64   
 12  cel                      231748 non-null  Int8    
 13  comptdr                  231748 non-null  In

### 10) Reconstrução categórica das variáveis escolaridade e ocupações

In [13]:
# Escolaridade
for col in ["escolaridade_pai", "escolaridade_mae"]:
    df_agg_21_23[col] = numero_para_categoria(
        df_agg_21_23[col],
        MAP_ESCOLARIDADE_REV,
        ORDEM_PAIS_ESCOLARIDADE
    )

# Ocupação
for col in ["ocup_pai", "ocup_mae"]:
    df_agg_21_23[col] = numero_para_categoria(
        df_agg_21_23[col],
        MAP_OCUP_REV,
        ORDEM_OCUPACAO
    )



### 11) Exportação


Ao final desta etapa, são salvos diferentes arquivos derivados da consolidação longitudinal de Minas Gerais, cada um com uma finalidade específica no pipeline analítico.

As saídas incluem:

* base histórica MG 2021–2023 em formato numérico
utilizada em análises quantitativas e como insumo para a preparação da base preditiva;

* base unificada de dados 2021–2024 (df_dados)
contendo as características socioeconômicas e demográficas harmonizadas ao longo dos anos;

* base unificada de resultados 2021–2024 (df_resultados)
contendo desempenho e indicadores de presença nas provas;

* base agregada socioeconômica (df_dados_agg)
usada em análises exploratórias e visualizações por grupo;

* base agregada de resultados (df_resultados_agg)
sintetizando desempenho médio, dispersão e indicadores de comparecimento;

* base integrada agregada (df_merged)
reunindo perfil socioeconômico e desempenho em nível agregado, especialmente importante para 2024, em que não há vínculo individual entre participantes e resultados;

* amostra estratificada da base de resultados (df_amostra_resultados)
construída a partir dos percentis de nota_media, útil para visualizações e análises complementares com menor custo computacional;

* base integrada numérica (df_merged_num)
voltada a análises quantitativas e preparação de insumos para modelagem;

* base agregada histórica 2021–2023 (df_agg_21_23)
preservando o período em que dados socioeconômicos e notas estavam originalmente na mesma base.

Esses arquivos servem de ponte entre as etapas de consolidação, exploração, dashboard e modelagem preditiva.

In [14]:
df_mg_21_23_num.to_parquet(DADOS_MG_21_23_NUM, index=False)
df_mg_21_23.to_parquet(DADOS_MG_21_23, index=False)
df_dados.to_parquet(DADOS_UNI_MG, index=False)
df_resultados.to_parquet(RESULTADOS_UNI_MG, index=False)
df_dados_agg.to_parquet(DADOS_AGG_MG, index=False)
df_resultados_agg.to_parquet(RESULTADOS_AGG_MG, index=False)
df_merged.to_parquet(MERGED_MG, index=False)
df_amostra_resultados.to_parquet(AMOSTRAG_RESULTADOS_MG, index=False)
df_agg_21_23.to_parquet(DADOS_AGG_MG_21_23, index=False)

print("✅ Salvo base mg 21/23 numérica:", DADOS_MG_21_23_NUM, "| shape:", df_mg_21_23_num.shape)
print("✅ Salvo base dados 21/24:", DADOS_UNI_MG, "| shape:", df_dados.shape)
print("✅ Salvo base resultados 21/24:", RESULTADOS_UNI_MG, "| shape:", df_resultados.shape)
print("✅ Salvo base dados agregados 21/24:", DADOS_AGG_MG, "| shape:", df_dados_agg.shape)
print("✅ Salvo base resultados agregados 21/24:", RESULTADOS_AGG_MG, "| shape:", df_resultados_agg.shape)
print("✅ Salvo dataframe merged MG 21/24:", MERGED_MG, "| shape:", df_merged.shape)
print("✅ Salvo amostra dos resultados 21/24:", AMOSTRAG_RESULTADOS_MG, "| shape:", df_amostra_resultados.shape)
print("✅ Salvo base mg 21/23 agregado:", DADOS_AGG_MG_21_23, "| shape:", df_agg_21_23.shape)

✅ Salvo base mg 21/23 numérica: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\mg_21_23_num.parquet | shape: (996185, 32)
✅ Salvo base dados 21/24: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\dados_uni_mg.parquet | shape: (1390009, 22)
✅ Salvo base resultados 21/24: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\resultados_uni_mg.parquet | shape: (1390009, 11)
✅ Salvo base dados agregados 21/24: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\dados_agg_mg.parquet | shape: (407079, 18)
✅ Salvo base resultados agregados 21/24: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\resultados_agg_mg.parquet | shape: (156, 24)
✅ Salvo dataframe merged MG 21/24: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\merged_mg.parquet | shape: (156, 32)
✅ Salvo amostra dos resultados 21/24: E:\ciencias_dados\projetos\projeto_enem_ml\dados\analiticos\resultados_amostrag_mg.parquet | shape: (246365, 11)
✅ Salvo base mg 21/23 agreg